In [1]:
from decouple import AutoConfig
config = AutoConfig(search_path='./../.env')

In [2]:
import os
import openai

openai.api_key = config('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = openai.api_key

## Functions, Models, and Chains

In [ ]:
!ollama pull "llama3.2"

In [4]:
from langchain_ollama.llms import OllamaLLM

os_llm =  OllamaLLM(model="llama3.2",
                         temperature=0.0,
                         max_tokens=1024,
                         top_k=10,)

### Function Binding

In [5]:
from langchain_openai import ChatOpenAI
model_name = 'gpt-4o-mini'
llm = ChatOpenAI(
        model=model_name,
        temperature=0.0,
        max_tokens=1024
    )

In [6]:
from langchain.prompts import ChatPromptTemplate
from langchain.schema import StrOutputParser

In [7]:
def joke_generator(topic):
    "This function will return a joke based on the topic provided"
    
    prompt = ChatPromptTemplate.from_template(
        "Write a short joke on {topic}"
    )
    output_parser = StrOutputParser()
    chain = prompt | os_llm | output_parser
    return chain.invoke({"topic":topic})
    

In [ ]:
response = joke_generator("Artificial Intelligence")
print(response)

In [9]:
def poem_generator(topic):
    "This function will return a poem based on the topic provided"
    prompt = ChatPromptTemplate.from_template(
        "Write a short poem on {topic}"
    )
    output_parser = StrOutputParser()
    chain = prompt | os_llm | output_parser
    return chain.invoke({"topic":topic})

In [ ]:
response = poem_generator("Artificial Intelligence")
print(response)

#### OpenAI functon format

In [11]:
functions =[
    {
      "name": "joke_generator",
      "description": "Generates a joke based on the topic provided",
      "parameters": {
        "type": "object",
        "properties": {
          "topic": {
            "type": "string",
            "description": "The topic to get the joke for"
          },
        },
        "required": ["topic"]
      }
    },
    {
      "name": "poem_generator",
      "description": "Generates a poem based on the topic provided",
      "parameters": {
        "type": "object",
        "properties": {
          "topic": {
            "type": "string",
            "description": "The topic to get the poem for"
          },
        },
        "required": ["topic"]
      }
    }
]

#### Attaching functions with model invocation

In [12]:
messages = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a smart and intelligent AI Assistant."),
        ("user", "{input}")
    ]
)

In [ ]:
llm.invoke(input="poem on Artificial Intelligence", functions=functions)

**NOTE:** With this approach, it is required to attach functions with everytime model is invoked.

#### Binding model with functions

In [14]:
llm_func = llm.bind(functions=functions)

In [15]:
runnable = messages | llm_func

In [ ]:
runnable.invoke({"input": "joke about Artificial Intelligence"})

In [ ]:
runnable.invoke({"input": "poem on Artificial Intelligence"})

## Tools and Models

In [18]:
from langchain_community.tools.arxiv.tool import ArxivQueryRun
from langchain_community.tools.pubmed.tool import PubmedQueryRun

arxiv_search = ArxivQueryRun()
pubmed_search = PubmedQueryRun()

tools = [arxiv_search, pubmed_search]

In [ ]:
arxiv_search.name

In [ ]:
arxiv_search.description

In [ ]:
print(arxiv_search({"query": "LLM Agents"}))

In [ ]:
from langchain.tools.render import format_tool_to_openai_function
format_tool_to_openai_function(arxiv_search)

In [23]:
functions = [
    format_tool_to_openai_function(f) for f in [
        arxiv_search, pubmed_search
    ]
]
model = llm.bind(function=functions)

In [24]:
tools = [arxiv_search, pubmed_search]
tools_model = llm.bind_tools(tools=tools)

In [ ]:
tools_model.invoke("survey on LLM Agents")

In [26]:
from langchain.agents.output_parsers import OpenAIFunctionsAgentOutputParser

In [27]:
from langchain.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful assistant"),
    ("user", "{input}"),
])
chain = prompt | tools_model | OpenAIFunctionsAgentOutputParser()

In [28]:
response = chain.invoke({"input": "survey on LLM Agents"})

In [ ]:
response